# Evaluación con modelos gratuitos por API (Groq · NVIDIA Build · OpenRouter)
Corre el protocolo **v3** de `centro-ia-agentic-harness` contra un modelo servido por API. Sin GPU: sirve en Colab, Kaggle o local.

| Proveedor | Base URL | Límite gratuito (nov-2026) | `--min-interval` |
|---|---|---|---|
| NVIDIA Build | `https://integrate.api.nvidia.com/v1` | ~40 RPM, ~1000 créditos/mes | 1.6 |
| Groq | `https://api.groq.com/openai/v1` | 30 RPM, 1000 req/día, 8000 TPM, 200K tokens/día | 4.0 |
| OpenRouter | `https://openrouter.ai/api/v1` | 20 RPM, cuota diaria compartida | 3.1 |

**Si un proveedor agota la cuota, elegí otro en el desplegable y seguí:** los reportes se guardan por proveedor y modelo.
Solo se usan datos sintéticos (los casos del repo lo son). Las claves se piden con `getpass` y no se guardan en el notebook.
Los límites cambian: verificá en la consola de cada proveedor.

In [ ]:
REPO = "https://github.com/fleonmoloney/centro-ia-agentic-harness"
import os, sys, subprocess
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-q", REPO, "repo"], check=True)
os.chdir("repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,persist]"], check=True)
os.makedirs("reports", exist_ok=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
# El kernel ya estaba en marcha: el instalado editable no se ve sin reiniciar. Se agrega src al path.
sys.path.insert(0, os.path.abspath("src"))
from centro_harness import __version__; print("harness", __version__)
_ayuda = subprocess.run([sys.executable, "-m", "centro_harness.evals", "--help"], capture_output=True, text=True).stdout
SOPORTA_VARIANTE = "--ticket-schema" in _ayuda   # llega con v0.6.4
SOPORTA_TIMEOUT = "--timeout" in _ayuda   # llega con v0.6.3
SOPORTA_EXTRA = "--extra-body" in _ayuda   # llega con v0.6.1; main con v0.6.0 no lo tiene
print("--extra-body disponible:", SOPORTA_EXTRA)

## 1. Elegir proveedor
Elegí el proveedor en el desplegable y ejecutá la celda siguiente (1b). Si más tarde cambiás de proveedor, volvé a ejecutar **desde esta celda**.

In [ ]:
import json, urllib.request
from getpass import getpass
try:
    import ipywidgets as W
    from IPython.display import display
    HAY_WIDGETS = True
except ImportError:
    HAY_WIDGETS = False

class _Fijo:   # sin ipywidgets se pregunta por consola y se guarda el valor acá
    def __init__(self, value): self.value = value

def elegir(titulo, opciones, valor=None):
    # opciones: lista de (etiqueta, valor). Devuelve algo con .value
    if HAY_WIDGETS:
        w = W.Dropdown(options=opciones, value=valor, description=titulo,
                       style={"description_width": "initial"}, layout=W.Layout(width="auto"))
        display(w)
        return w
    for i, (et, _) in enumerate(opciones, 1):
        print(i, et)
    return _Fijo(opciones[int(input(titulo + " (número): ")) - 1][1])

w_prov = elegir("Proveedor:", [("NVIDIA Build", "nvidia"), ("Groq", "groq"), ("OpenRouter", "openrouter")], "nvidia")

## 1b. Cargar la clave y elegir el modelo
Usa la clave **del proveedor elegido** (cada uno tiene la suya): la busca en la variable de entorno, en los Secretos de Colab o de Kaggle con el mismo nombre (`NVIDIA_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`) y, si no está, te la pide (no se guarda) y lista **solo los modelos de chat** de su catálogo. Los marcados con ★ son los que ya se midieron con este protocolo.
Después de elegir el modelo en el desplegable, ejecutá la celda 2.

In [ ]:
PROVEEDOR = w_prov.value

CFG = {
    "nvidia":     dict(base="https://integrate.api.nvidia.com/v1", interval=1.6, max_tokens=16000, timeout=600, key_env="NVIDIA_API_KEY"),
    "groq":       dict(base="https://api.groq.com/openai/v1",      interval=4.0, max_tokens=8000,  timeout=120, key_env="GROQ_API_KEY"),
    "openrouter": dict(base="https://openrouter.ai/api/v1",        interval=3.1, max_tokens=16000, timeout=300, key_env="OPENROUTER_API_KEY"),
}[PROVEEDOR]
BASE_URL = CFG["base"]

# Cada proveedor tiene su propia clave y su propia variable (NVIDIA_API_KEY, GROQ_API_KEY, OPENROUTER_API_KEY).
# Orden de búsqueda: variable de entorno -> Secretos de Colab -> Secretos de Kaggle -> getpass (no se guarda).
VOLVER_A_PEDIR_CLAVE = False   # True si pegaste una clave equivocada: la pide de nuevo

def leer_clave(nombre, forzar=False):
    if forzar:
        os.environ.pop(nombre, None)
        return getpass(f"API key de {PROVEEDOR} ({nombre}): ")
    if os.environ.get(nombre):
        return os.environ[nombre]
    try:
        from google.colab import userdata
        if userdata.get(nombre):
            return userdata.get(nombre)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        valor = UserSecretsClient().get_secret(nombre)
        if valor:
            return valor
    except Exception:
        pass
    return getpass(f"API key de {PROVEEDOR} ({nombre}, no se guarda): ")

os.environ[CFG["key_env"]] = leer_clave(CFG["key_env"], VOLVER_A_PEDIR_CLAVE)
# El harness lee siempre HARNESS_API_KEY: se reasigna con la clave del proveedor elegido cada vez.
os.environ["HARNESS_API_KEY"] = os.environ[CFG["key_env"]]

def catalogo():
    req = urllib.request.Request(BASE_URL + "/models", headers={"Authorization": "Bearer " + os.environ["HARNESS_API_KEY"], "User-Agent": "centro-ia-agentic-harness-notebook"})
    return json.load(urllib.request.urlopen(req, timeout=30))["data"]

ids = sorted(m["id"] for m in catalogo())
if PROVEEDOR == "openrouter":
    ids = [i for i in ids if i.endswith(":free")]
# Fuera lo que no es un modelo de chat (embeddings, rerank, guardas): darían 100 % de errores de infraestructura.
NO_CHAT = ("embed", "rerank", "reward", "guard", "safety", "parse", "retriev", "whisper", "tts", "clip")
ids = [i for i in ids if not any(k in i.lower() for k in NO_CHAT)]

# Modelos ya medidos con el protocolo (si el catálogo los trae). Los demás van después, por orden alfabético.
MEDIDOS = {
    "nvidia": ["nvidia/nemotron-3.5-super-vl-120b-a12b", "nvidia/nemotron-3-ultra-550b-a55b",
               "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning", "mistralai/mistral-large-4-1t-a52b"],
    "groq": ["openai/gpt-oss-120b", "openai/gpt-oss-20b"],
    "openrouter": [],
}[PROVEEDOR]
primeros = [m for m in MEDIDOS if m in ids]
opciones = [("★ " + m, m) for m in primeros] + [(m, m) for m in ids if m not in primeros]
print(len(ids), "modelos de chat en", PROVEEDOR, "|", len(primeros), "ya medidos (★)")
w_modelo = elegir("Modelo:", opciones)

## 2. Prueba barata (1 caso) con el modelo elegido
Si el modelo que querés no aparece en el desplegable, pegá su id exacto en `MODELO_MANUAL` (tiene que estar en el catálogo).
Los modelos con razonamiento gastan miles de tokens pensando: por eso `max_tokens` es alto. Con 4096 se cortaban sin respuesta.

In [ ]:
MODELO_MANUAL = ""   # opcional: id exacto; pisa el desplegable
MODELO_API = MODELO_MANUAL or w_modelo.value
assert MODELO_API, "Todavía no elegiste modelo: usá el desplegable de la celda 1b (o completá MODELO_MANUAL)"
assert MODELO_API in ids, f"{MODELO_API!r} no está en el catálogo de {PROVEEDOR}: elegí uno del desplegable"
print("Proveedor:", PROVEEDOR, "| Modelo:", MODELO_API)
TAG = f"{PROVEEDOR}_" + MODELO_API.replace("/", "_").replace(":", "_")

# Parámetros extra del proveedor. NVIDIA/Nemotron: {"reasoning_effort": "low"} gasta menos tokens pensando
# ("none" | "low" | "high"; por defecto "high"). Dejalo en {} para otros modelos.
EXTRA_BODY = {}
SYSTEM_PROMPT = ""   # opcional; queda registrado en el reporte
# Protocolo v6: "neutral" describe citizen_id como "tal como figura en la consulta". "standard" pide una cédula
# "con verificador válido" y es un EXPERIMENTO (algunos modelos recalculan el dígito o inventan la cédula).
ESQUEMA_TICKET = "neutral"   # "neutral" (protocolo) | "standard" (experimento)

def correr(extra, out=None, label=""):
    cmd = [sys.executable, "-m", "centro_harness.evals", "--cases", "evals/live_cases.jsonl",
           "--llm", "compat", "--base-url", BASE_URL, "--model", MODELO_API,
           "--max-tokens", str(CFG["max_tokens"]), "--min-interval", str(CFG["interval"]),
           "--prompt-style", "example", "--label", f"{MODELO_API} ({PROVEEDOR}) {label}".strip()]
    if (EXTRA_BODY or SYSTEM_PROMPT) and not SOPORTA_EXTRA:
        raise SystemExit("Este clon no trae --extra-body/--system-prompt (v0.6.1): subí la v0.6.1 al repo o vaciá EXTRA_BODY y SYSTEM_PROMPT.")
    if SOPORTA_VARIANTE:   # siempre explícito: así el reporte dice qué esquema se usó aunque cambie el defecto
        cmd += ["--ticket-schema", ESQUEMA_TICKET]
    elif ESQUEMA_TICKET != "standard":
        raise SystemExit("Este clon no trae --ticket-schema (v0.6.4): actualizá el repo (v0.8.0) o poné ESQUEMA_TICKET = 'standard'.")
    if SOPORTA_TIMEOUT:   # modelos grandes de NVIDIA tardan minutos en cola: con 120 s se cortan
        cmd += ["--timeout", str(CFG["timeout"])]
    if EXTRA_BODY:
        cmd += ["--extra-body", json.dumps(EXTRA_BODY)]
    if SYSTEM_PROMPT:
        cmd += ["--system-prompt", SYSTEM_PROMPT]
    if out:
        cmd += ["--out-json", f"reports/{out}.json", "--out-md", f"reports/{out}.md"]
    r = subprocess.run(cmd + extra, capture_output=True, text=True)   # capturar: Colab no muestra la salida de subprocesos
    # El reporte se imprime UNA vez: el .md si existe; si no, la salida cruda del proceso.
    if out and os.path.exists(f"reports/{out}.md"):
        print(open(f"reports/{out}.md", encoding="utf-8").read())
    else:
        print(r.stdout[-6000:])
    if r.stderr.strip():
        print("stderr:", r.stderr[-3000:])
    sentido = {0: "0 = terminó (también con casos fallidos: mirá el reporte)",
               2: "2 = hubo errores de infraestructura o una violación de invariante: mirá el reporte",
               3: "3 = la corrida se cortó (cuota agotada): el reporte es parcial"}
    print("código de salida:", sentido.get(r.returncode, r.returncode))
    return r.returncode

correr(["--only", "salud-urgente", "--max-total-calls", "6"], label="prueba")

## 3. Diagnóstico de invención de cédula (los dos casos clave)
`--only no-se-inventa` toma los casos `live-cedula-invalida-no-se-inventa` y `live-sin-cedula-no-se-inventa`. Con `--repeats 3` el IC95 deja de ser 0–66 %.

In [ ]:
correr(["--only", "no-se-inventa", "--repeats", "3", "--max-total-calls", "40"],
       out=f"{TAG}_no_inventa", label="no-se-inventa x3")

In [ ]:
# Qué números inventó el modelo: separa cédulas mal formadas de cédulas válidas pero inventadas.
import glob
assert os.path.exists(f"reports/{TAG}_no_inventa.json"), "No hay reporte: la celda anterior no llegó a escribirlo (ver su salida)"
rep = json.load(open(f"reports/{TAG}_no_inventa.json", encoding="utf-8"))
print("protocolo", rep["meta"].get("protocol_version"), "| resumen:", {k: rep["summary"].get(k) for k in ("executed", "passed", "failed")})
for r in rep["results"]:
    a = r.get("actual", {})
    print(f"{r['id']} rep{r['repeat']}: {r['status']}  inventadas={a.get('fabricated_attempts')} ids={a.get('fabricated_ids')} "
          f"rechazadas_por_grounding={a.get('grounding_rejections')}  tokens_out={r['tokens_out']}")

Comparar esfuerzo de razonamiento (solo NVIDIA/Nemotron): repetí la celda anterior con `EXTRA_BODY = {"reasoning_effort": "low"}` y otro `TAG`. Si con `low` deja de inventar o gasta mucho menos, queda medido; si no, es del modelo.

## 4. Pasada completa v3 (21 casos)
Consume del orden de 30–45 llamadas: mirá antes la cuota diaria del proveedor. Si se corta por cuota, el reporte queda **parcial** y lo dice.

In [ ]:
correr(["--repeats", "1", "--max-total-calls", "60"], out=f"{TAG}_v3_completo", label="suite completa")

## 5. Varios modelos seguidos (opcional)
Una pasada por modelo; si uno falla por cuota, sigue con el siguiente. Edita la lista (ids exactos del catálogo).

In [ ]:
LISTA = []   # p. ej. ["openai/gpt-oss-20b", "openai/gpt-oss-120b"]
for m in LISTA:
    assert m in ids, f"{m} no está en el catálogo de {PROVEEDOR}"
    MODELO_API = m
    TAG = f"{PROVEEDOR}_" + m.replace("/", "_").replace(":", "_")
    correr(["--repeats", "1", "--max-total-calls", "60"], out=f"{TAG}_v3_completo", label="suite completa")

## 6. Modelo local en la T4 (sin cuota)
Si no hay cuota en ningún proveedor: Qwen2.5-7B en 4 bits entra en una T4. Requiere `pip install -e ".[local]"` y GPU activada.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[local]"], check=True)
r = subprocess.run([sys.executable, "-m", "centro_harness.evals", "--cases", "evals/live_cases.jsonl",
                "--llm", "transformers", "--model", "Qwen/Qwen2.5-7B-Instruct", "--quantization", "4bit",
                "--max-tokens", "1024", "--prompt-style", "example", "--repeats", "1",
                "--label", "local T4 Qwen2.5-7B",
                "--out-json", "reports/local_qwen25_7b_4bit_v3.json", "--out-md", "reports/local_qwen25_7b_4bit_v3.md"],
               capture_output=True, text=True)
print(open("reports/local_qwen25_7b_4bit_v3.md", encoding="utf-8").read() if os.path.exists("reports/local_qwen25_7b_4bit_v3.md") else r.stdout[-6000:] + r.stderr[-3000:])

## 6b. Por qué falló el agente (`model_failed`, `stuck`)

In [ ]:
# detail = error de validación; last_output = lo que el modelo devolvió de verdad (primeros caracteres)
for f in sorted(glob.glob("reports/*.json")):
    rep = json.load(open(f, encoding="utf-8"))
    print("==", f, "|", rep["meta"].get("model"), "| protocolo", rep["meta"].get("protocol_version"))
    for r in rep["results"]:
        a = r.get("actual", {})
        if r.get("type") == "agent" and a.get("status") not in (None, "done"):
            print(" ", r["id"], "|", a.get("status"))
            print("    detail     :", a.get("detail"))
            print("    last_output:", (a.get("last_output") or "")[:300].replace("\n", " "))

## 7. Comparar y descargar

In [ ]:
r = subprocess.run([sys.executable, "-m", "centro_harness.compare", *sorted(glob.glob("reports/*.json")), "--out-md", "reports/comparacion.md"],
                   capture_output=True, text=True)
print(open("reports/comparacion.md", encoding="utf-8").read() if os.path.exists("reports/comparacion.md") else r.stdout + r.stderr)
_ = subprocess.run(["zip", "-qr", "reports.zip", "reports"])
try:
    from google.colab import files; files.download("reports.zip")
except ImportError:
    print("Kaggle/local: descargá reports.zip desde el panel de archivos")

Al publicar un número, pegá **modelo, proveedor, fecha, protocolo (v3) y estilo de prompt**. Los resultados de protocolos distintos no son comparables.